# Matched stars outside M67

This notebook explains and runs the expanded **catalog-based** pipeline. It uses measured Hubble Source Catalog detections, Gaia positions, and LAMOST spectra. It does not search for planets. Run the cells in order. Existing cached inputs are reused.

The saved result contains 31 stars outside M67; three earlier M67 matches bring the total to 34. Read README.txt for quality warnings and selection limits.

In [ ]:
from pathlib import Path
import json, subprocess, sys

# Locate the pipeline whether Jupyter starts here or in the project folder.
ROOT = Path.cwd()
if not (ROOT / "matches.json").exists():
    ROOT = ROOT / "outputs" / "matched_star_survey"
assert (ROOT / "matches.json").exists(), "Open this notebook from its folder or the project folder."
ROOT = ROOT.resolve()

def run_step(filename):
    """Run one saved pipeline script with the notebook Python environment."""
    subprocess.run([sys.executable, str(ROOT / filename)], check=True, cwd=ROOT)


## 1. Search the catalogs
`eligible_rows()` selects usable LAMOST parameter records and one observation per Gaia star. `search_field()` finds nearby Hubble catalog entries. `fetch()` caches archive responses. The wide discovery radius does not itself prove a match.

In [ ]:
# Network needed only for missing archive responses.
run_step("search_fields.py")

## 2. Check Hubble positions against Gaia
`gaia_sources()` retrieves exact Gaia identifiers. `position_at()` moves each Gaia position to the Hubble observation date. `details()` retrieves per-image Hubble measurements. `main()` requires a consistent, unique catalog association and saves accepted/rejected candidates.

In [ ]:
run_step("validate_matches.py")

## 3. Check nearby stars and LAMOST positions
`check()` rejects a match if another Gaia star is too close to the Hubble detection, or if the LAMOST coordinates are closer to a different Gaia star. This final check removed three preliminary candidates in this run.

In [ ]:
run_step("check_neighbors.py")

## 4. Download the evidence
`cutout()` downloads a small real Hubble FITS image. `fetch_spectrum()` downloads an exact LAMOST observation. `extract_spectrum()` reads wavelength, flux, errors and masks. `retrieve()` joins catalog properties to those spectra and preserves warnings. Image cutouts are visual evidence, not new centroid fits.

In [ ]:
run_step("download_evidence.py")

## 5. Verify and summarize
`make_report.py` checks unique IDs, complete spectra/cutouts, and at least 30 matches. It creates the combined catalog, simple text table and preview images.

In [ ]:
run_step("make_report.py")
summary = json.loads((ROOT / "summary.json").read_text())
summary

## 6. View the stars and measurements
Zero quality flags do not guarantee perfect data. A positional match does not establish a planet, habitability, or cluster membership.

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / "hubble_contact_sheet.png"), width=1000))
stars = json.loads((ROOT / "star_catalog.json").read_text())
first = stars[0]
{k: first["stellar_properties"][k] for k in ["temperature_K", "Fe_H_dex", "logg_dex", "fibermask"]}